# Lab 6: Word Sense Disambiguation (full version)

Disambiguate the words of the *SMTeuroparl* sentence pairs (test set) with WordNet, compute the similarity of each pair using senses and the Jaccard distance, and compare it with the words (Lab 2) and lemmas (Lab 3) approaches using the Pearson correlation with the gold standard.

This is the extended version of the notebook, for study and not for submission (the submitted one is in the team repository `team/`, tag `lab06-submitted`). It brings together everything found while working on this lab:

- **Three pipelines**: NLTK (`word_tokenize` → `pos_tag` → `WordNetLemmatizer`), spaCy (`nlp`) and FreeLing (TextServer). Each pipeline gives its own words, lemmas and POS tags, and the senses are computed from them, so the columns of the results never mix libraries.
- **Three WSD algorithms**: Lesk (`nltk.wsd.lesk`) and UKB (TextServer), from the statement, and MFS (the first WordNet synset, the usual baseline). Lesk and MFS are run on the three pipelines. UKB only exists with FreeLing, because TextServer gives its senses together with its tokens.

The design decisions taken along the way are collected in the last section (*Notes*).


## Setup

Import the libraries, download the NLTK packages (tokenizer, POS tagger, WordNet) and the spaCy model, and import the TextServer client (`textserver.py` in `common/`).


In [1]:
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
if "google.colab" in sys.modules:
    %pip install -q nltk spacy pandas scipy
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

DATA_DIR = ROOT / "test_gold"
NLTK_DIR = ROOT / "nltk_data"
COMMON_DIR = ROOT / "common"
UKB_CACHE = COMMON_DIR / "ukb_senses.json"

# Notebook libs
import json
from collections import Counter
from getpass import getpass

import nltk
import pandas as pd
import requests
import spacy
from nltk.corpus import wordnet as wn
from nltk.metrics.distance import jaccard_distance
from nltk.stem import WordNetLemmatizer
from nltk.wsd import lesk
from scipy.stats import pearsonr

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR)
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)

# TextServer client (textserver.py in common/), used for UKB
sys.path.insert(0, str(COMMON_DIR))
from textserver import TextServer

# TextServer with the user and password of credentials.json in common/
# or asked for if it is not there.
def textserver_login(service):
    credentials_file = COMMON_DIR / "credentials.json"
    if credentials_file.exists():
        credentials = json.loads(credentials_file.read_text())
    else:
        credentials = {"user": input("TextServer user: "), "password": getpass("TextServer password: ")}
    return TextServer(credentials["user"], credentials["password"], service)

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Data and preprocessing

Load the *SMTeuroparl* sentence pairs with their gold-standard scores, and process every sentence with the NLTK and spaCy pipelines (FreeLing comes from TextServer, in the next section). The preprocessing has three steps, and each one needs the previous one:

1. **Tokens**: split the sentence into words.
2. **POS tags**: the category of each word. Only nouns, verbs, adjectives and adverbs can have a WordNet sense, and the tag chooses which WordNet category is looked up.
3. **Lemmas**: the base form of each word. The lemmatizer needs the POS tag (*leaves* is *leaf* as a noun and *leave* as a verb).

spaCy does the three steps in one call (`nlp`). NLTK needs one function per step, and its Penn Treebank tags (`NN`, `VBZ`...) are mapped to the WordNet categories. To use the same code for all the pipelines, every sentence is stored as a list of `(word, lemma, WordNet POS)`, where the WordNet POS is `None` for the words that cannot have a sense.

In [2]:
# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / "STS.input.SMTeuroparl.txt", sep="\t", header=None)
dt["gs"] = pd.read_csv(DATA_DIR / "STS.gs.SMTeuroparl.txt", sep="\t", header=None)
print(f"Sentence pairs: {len(dt)}")

# spaCy: tokens, POS tags and lemmas in one pass.
nlp = spacy.load("en_core_web_sm")
spacy_pos = {"NOUN": wn.NOUN, "PROPN": wn.NOUN, "VERB": wn.VERB, "ADJ": wn.ADJ, "ADV": wn.ADV}

def spacy_tokens(sentence):
    return [(t.text, t.lemma_, spacy_pos.get(t.pos_)) for t in nlp(sentence)]

# NLTK: tokenizer, then Penn Treebank tags (NN* nouns, VB* verbs, JJ* adjectives, RB* adverbs),
# then the WordNet lemmatizer with the POS tag. The other words keep their form as lemma.
penn_pos = {"NN": wn.NOUN, "VB": wn.VERB, "JJ": wn.ADJ, "RB": wn.ADV}
wnl = WordNetLemmatizer()

def nltk_tokens(sentence):
    tokens = []
    for word, tag in nltk.pos_tag(nltk.word_tokenize(sentence)):
        pos = penn_pos.get(tag[:2])
        tokens.append((word, wnl.lemmatize(word, pos) if pos else word, pos))
    return tokens

pipelines = ["NLTK", "spaCy"]
tokens1 = {"NLTK": [nltk_tokens(s) for s in dt[0]], "spaCy": [spacy_tokens(s) for s in dt[0]]}
tokens2 = {"NLTK": [nltk_tokens(s) for s in dt[1]], "spaCy": [spacy_tokens(s) for s in dt[1]]}

Sentence pairs: 459


### FreeLing (TextServer)

The TextServer `senses` service does its own tokenization, POS tagging and lemmatization (FreeLing) and runs UKB in the same call, so its tokens are the third pipeline. For every word it returns the form, the lemma, the tag, the POS and the sense.

Each distinct sentence is sent once to TextServer. The answers are saved in `common/ukb_senses.json`, so the server is only queried the first time the notebook is run. With this file, it runs without TextServer.


In [3]:
# Query TextServer once for every distinct sentence, and keep the answers in a JSON file.
sentences = sorted(set(dt[0]) | set(dt[1]))
ukb_raw = json.loads(UKB_CACHE.read_text()) if UKB_CACHE.exists() else {}
missing = [s for s in sentences if s not in ukb_raw]
if missing:
    ts = textserver_login("senses")
    for sentence in missing:
        # The server sometimes fails or does not answer, so try up to 3 times.
        for attempt in range(3):
            try:
                answer = ts.senses(sentence)
                break
            except requests.RequestException:
                if attempt == 2:
                    raise
        # TextServer may split a sentence in two, so join the tokens of all its sentences.
        ukb_raw[sentence] = [token for sent in answer for token in sent]
        # Save after every query, so an interrupted run does not lose the answers.
        UKB_CACHE.write_text(json.dumps(ukb_raw, ensure_ascii=False, indent=1))
print(f"Distinct sentences: {len(sentences)}   queried now: {len(missing)}")

# FreeLing: the tokens, lemmas and POS tags given by TextServer with the senses.
freeling_pos = {"noun": wn.NOUN, "verb": wn.VERB, "adjective": wn.ADJ, "adverb": wn.ADV}

def freeling_tokens(sentence):
    return [(form, lemma, freeling_pos.get(pos)) for form, lemma, tag, pos, wn_id in ukb_raw[sentence]]

pipelines.append("FreeLing")
tokens1["FreeLing"] = [freeling_tokens(s) for s in dt[0]]
tokens2["FreeLing"] = [freeling_tokens(s) for s in dt[1]]

# Example: the first sentence with the three pipelines.
print(dt[0][0])
pd.concat({pipe: pd.DataFrame(tokens1[pipe][0], columns=["word", "lemma", "WordNet POS"]) for pipe in pipelines}, axis=1)

Distinct sentences: 338   queried now: 0
The leaders have now been given a new chance and let us hope they seize it.


NLTK                        spaCy                     FreeLing          \
       word   lemma WordNet POS     word   lemma WordNet POS     word   lemma   
0       The     The         NaN      The     the         NaN      The     the   
1   leaders  leader           n  leaders  leader           n  leaders  leader   
2      have    have           v     have    have         NaN     have    have   
3       now     now           r      now     now           r      now     now   
4      been      be           v     been      be         NaN     been      be   
5     given    give           v    given    give           v    given    give   
6         a       a         NaN        a       a         NaN        a       a   
7       new     new           a      new     new           a      new     new   
8    chance  chance           n   chance  chance           n   chance  chance   
9       and     and         NaN      and     and         NaN      and     and   
10      let     let           v      let     let           v      let     let   
11       us      us         NaN       us      we         NaN       us      us   
12     hope    hope           v     hope    hope           v     hope    hope   
13     they    they         NaN     they    they         NaN     they    they   
14    seize   seize           v    seize   seize           v    seize   seize   
15       it      it         NaN       it      it         NaN       it      it   
16        .       .         NaN        .       .         NaN        .       .   

                
   WordNet POS  
0          NaN  
1            n  
2            v  
3            r  
4            v  
5            v  
6          NaN  
7            a  
8            n  
9          NaN  
10           v  
11         NaN  
12           v  
13         NaN  
14           v  
15         NaN  
16         NaN

### POS tags

The three pipelines tokenize this sentence in the same way, but the POS tags differ: *have* and *been* are verbs for NLTK and FreeLing, and auxiliaries without a WordNet category for spaCy. A wrong tag looks up the wrong WordNet category, and the lemmatizer cannot reduce the word. For example, in pair 12 (used in Lab 3) the NLTK tagger tags *adopts* as a plural noun.

In [4]:
# Pair 12, second sentence: Penn Treebank tags of NLTK vs. spaCy tags.
sentence = dt[1][12]
print(sentence)
print("NLTK: ", nltk.pos_tag(nltk.word_tokenize(sentence)))
print("spaCy:", [(t.text, t.pos_) for t in nlp(sentence)])
print()
print("NLTK: ", tokens2["NLTK"][12])
print("spaCy:", tokens2["spaCy"][12])

(The Parliament adopts legislative resolution)
NLTK:  [('(', '('), ('The', 'DT'), ('Parliament', 'NNP'), ('adopts', 'NNS'), ('legislative', 'JJ'), ('resolution', 'NN'), (')', ')')]
spaCy: [('(', 'PUNCT'), ('The', 'DET'), ('Parliament', 'PROPN'), ('adopts', 'VERB'), ('legislative', 'ADJ'), ('resolution', 'NOUN'), (')', 'PUNCT')]

NLTK:  [('(', '(', None), ('The', 'The', None), ('Parliament', 'Parliament', 'n'), ('adopts', 'adopts', 'n'), ('legislative', 'legislative', 'a'), ('resolution', 'resolution', 'n'), (')', ')', None)]
spaCy: [('(', '(', None), ('The', 'the', None), ('Parliament', 'Parliament', 'n'), ('adopts', 'adopt', 'v'), ('legislative', 'legislative', 'a'), ('resolution', 'resolution', 'n'), (')', ')', None)]


### Case of the lemmas

The two lemmatizers treat the case differently. The spaCy lemmas are lowercased, except for proper nouns and *I*. The NLTK lemmatizer does not change the case (*The* stays *The*). The FreeLing lemmas are lowercased, except for document codes such as *A5-0241/2000*. The lemmas are used as each library gives them, without extra lowercasing.

The case does not matter for the senses, because WordNet lowercases the word before looking it up.

In [5]:
# Tokens whose lemma is not lowercase, in each pipeline.
for pipe in pipelines:
    capitalized = Counter((lemma, pos) for tokens in [tokens1[pipe], tokens2[pipe]] for sentence in tokens
                          for _, lemma, pos in sentence if lemma != lemma.lower())
    print(f"{pipe:<8} tokens with a capitalized lemma: {sum(capitalized.values())}   most frequent:",
          ", ".join(f"{lemma} {n}" for (lemma, _), n in capitalized.most_common(6)))

# WordNet lowercases the word before looking it up, so the case does not change the senses.
print("\nSame synsets for 'Report' and 'report':", wn.synsets("Report", wn.NOUN) == wn.synsets("report", wn.NOUN))

NLTK     tokens with a capitalized lemma: 1627   most frequent: The 185, I 119, It 75, Mr 66, President 66, We 61
spaCy    tokens with a capitalized lemma: 863   most frequent: I 136, Mr 66, President 66, A5 64, Parliament 53, Europe 42
FreeLing tokens with a capitalized lemma: 64   most frequent: A5-0323/2000 22, A5-0241/2000 22, A5-0323 10, A5-0241 10

Same synsets for 'Report' and 'report': True


## Word sense disambiguation

Every word with a WordNet category is disambiguated with three algorithms:

- **Lesk** (`nltk.wsd.lesk`): chooses the synset whose definition shares the most words with the sentence (the context is the whole sentence).
- **MFS** (most frequent sense): WordNet lists the senses of a word from the most to the least frequent, so MFS is the first one. It does not look at the context.
- **UKB** (TextServer): runs Personalized PageRank over the WordNet graph, starting from the senses of all the words of the sentence. Its synsets are returned as WordNet 3.0 offsets (`10285313-n`), which are converted to NLTK synsets.

For Lesk and MFS, the word looked up is the lemma, or the word itself if the lemma is not in WordNet. The lemmatizer matters less than the POS tag here, because `wn.synsets` also reduces the word (*adopts* → `adopt.v.01`); what changes the sense is the category given by the POS tag.

In [6]:
# For every pipeline and algorithm: (sentences 1, sentences 2),
# where each sentence is a list of (word, lemma, synset), and the synset is None when the word has no sense.
algorithms = ["Lesk", "MFS", "UKB"]
senses = {(pipe, a): ([], []) for pipe in pipelines for a in ["Lesk", "MFS"]}

for pipe in pipelines:
    for side, tokens in enumerate([tokens1[pipe], tokens2[pipe]]):
        for sentence in tokens:
            # Lesk uses the whole sentence as context.
            context = [word for word, _, _ in sentence]
            lesk_sentence, mfs_sentence = [], []
            for word, lemma, pos in sentence:
                # Look up the lemma, or the word itself if the lemma is not in WordNet.
                key = lemma if pos and wn.synsets(lemma, pos) else word
                synsets = wn.synsets(key, pos) if pos else []
                if synsets:
                    # The candidate synsets are given to Lesk, because lesk(context, key, pos)
                    # drops the satellite adjectives (see the next cell).
                    lesk_ss = lesk(context, key, synsets=synsets)
                    mfs_ss = synsets[0]
                else:
                    lesk_ss, mfs_ss = None, None
                lesk_sentence.append((word, lemma, lesk_ss))
                mfs_sentence.append((word, lemma, mfs_ss))
            senses[pipe, "Lesk"][side].append(lesk_sentence)
            senses[pipe, "MFS"][side].append(mfs_sentence)

# '10285313-n' -> Synset('male_child.n.01'), or None for 'N/A'.
def to_synset(wn_id):
    if wn_id == "N/A":
        return None
    offset, pos = wn_id.split("-")
    return wn.synset_from_pos_and_offset(pos, int(offset))

# UKB only exists on the FreeLing tokens, because TextServer gives them together.
senses["FreeLing", "UKB"] = tuple([[(form, lemma, to_synset(wn_id)) for form, lemma, tag, pos, wn_id in ukb_raw[s]] for s in dt[side]]
                                  for side in (0, 1))

# Example: the senses of the first sentence (FreeLing pipeline, the same tokens for the three algorithms).
print(dt[0][0])
rows = []
for (word, lemma, lesk_ss), (_, _, mfs_ss), (_, _, ukb_ss) in zip(*[senses["FreeLing", a][0][0] for a in algorithms]):
    if ukb_ss or lesk_ss:
        rows.append({"word": word, "lemma": lemma, "Lesk": lesk_ss and lesk_ss.name(), "MFS": mfs_ss and mfs_ss.name(),
                     "UKB": ukb_ss and ukb_ss.name()})
pd.DataFrame(rows).set_index("word")

The leaders have now been given a new chance and let us hope they seize it.


,lemma,Lesk,MFS,UKB
word,,,,
leaders,leader,leader.n.01,leader.n.01,leader.n.01
have,have,have.v.01,have.v.01,get.v.03
now,now,now.r.01,now.r.01,immediately.r.01
been,be,be.v.01,be.v.01,be.v.01
given,give,give.v.05,give.v.01,hold.v.03
new,new,fresh.s.03,new.a.01,new.a.06
chance,chance,probability.n.01,opportunity.n.01,opportunity.n.01
let,let,let.v.01,let.v.01,get.v.03
hope,hope,hope.v.01,hope.v.01,hope.v.02


### Lesk and the satellite adjectives

`nltk.wsd.lesk(context, word, pos)` keeps only the synsets whose category is exactly `pos`. But most WordNet adjectives are *satellites* (category `s`, not `a`), so with `pos="a"` they are all dropped and Lesk returns `None`, while MFS (`wn.synsets(word, "a")`, which includes them) finds a sense. Giving Lesk the candidate synsets avoids this.

The NLTK implementation is also very simple: it splits the definition on spaces, does not lowercase or lemmatize it, does not remove stopwords, does not use the examples, and returns the first synset when there is a tie (so with no overlap at all it returns the first sense, like MFS).

In [7]:
# The adjective "fervent" only has satellite synsets.
synsets = wn.synsets("fervent", wn.ADJ)
context = "We are fervent supporters of the report .".split()
print("Synsets:", synsets, [s.pos() for s in synsets])
print("lesk(context, word, pos):    ", lesk(context, "fervent", wn.ADJ))
print("lesk(context, word, synsets):", lesk(context, "fervent", synsets=synsets))

# Number of words that would have no Lesk sense with lesk(context, key, pos), in the spaCy pipeline.
lost = Counter()
for sentence in tokens1["spaCy"] + tokens2["spaCy"]:
    context = [word for word, _, _ in sentence]
    for word, lemma, pos in sentence:
        key = lemma if pos and wn.synsets(lemma, pos) else word
        if pos and wn.synsets(key, pos) and lesk(context, key, pos) is None:
            lost[key] += 1
print(f"\nWords without a Lesk sense when using pos: {sum(lost.values())}   most frequent:",
      ", ".join(f"{w} {n}" for w, n in lost.most_common(8)))

Synsets: [Synset('ardent.s.01'), Synset('fervent.s.02')] ['s', 's']
lesk(context, word, pos):     None
lesk(context, word, synsets): Synset('ardent.s.01')

Words without a Lesk sense when using pos: 146   most frequent: transferable 31, fervent 19, selective 17, such 17, unanimous 17, inherent 17, latent 15, ardent 5


## Similarities and Pearson correlation

For every pipeline, Jaccard similarity of each pair using:

- **words** and **lemmas** (Lab 2 and Lab 3);
- **senses**: only the synsets of the disambiguated words;
- **senses + rest**: the synsets, plus the lemma of the words without a sense (function words, punctuation, numbers, words not in WordNet). It only differs from lemmas in the words that get a sense, so it separates the effect of the disambiguation from the effect of leaving out the words without a sense.

The NLTK column is the one that matches Lab 2 (NLTK words); Lab 3 used spaCy lemmas as its main result. FreeLing is the only column with UKB, and it also has Lesk and MFS, so the three algorithms can be compared on the same tokens.

In [8]:
def jaccard(set1, set2):
    # Two empty sets share nothing (with UKB, pair 260 has no sense in either sentence).
    if not set1 and not set2:
        return 0.0
    return 1 - jaccard_distance(set1, set2)

def sense_set(sentence, keep_rest):
    """Synsets of the disambiguated words, plus (if keep_rest) the lemma of the other words."""
    items = set()
    for _, lemma, synset in sentence:
        if synset:
            items.add(synset.name())
        elif keep_rest:
            items.add(lemma)
    return items

for pipe in pipelines:
    pairs = list(zip(tokens1[pipe], tokens2[pipe]))
    dt[f"{pipe} words"] = [jaccard({w for w, _, _ in x1}, {w for w, _, _ in x2}) for x1, x2 in pairs]
    dt[f"{pipe} lemmas"] = [jaccard({l for _, l, _ in x1}, {l for _, l, _ in x2}) for x1, x2 in pairs]

for (pipe, a), (s1, s2) in senses.items():
    dt[f"{pipe} {a} senses"] = [jaccard(sense_set(x1, False), sense_set(x2, False)) for x1, x2 in zip(s1, s2)]
    dt[f"{pipe} {a} senses + rest"] = [jaccard(sense_set(x1, True), sense_set(x2, True)) for x1, x2 in zip(s1, s2)]

representations = ["words", "lemmas"] + [f"{a} senses{rest}" for a in algorithms for rest in ["", " + rest"]]

# One value for each representation and pipeline (NaN for UKB, which only exists with FreeLing).
def table(value):
    return pd.DataFrame({pipe: [value(dt[f"{pipe} {r}"]) if f"{pipe} {r}" in dt else float("nan") for r in representations]
                         for pipe in pipelines}, index=representations)

pd.concat({"Pearson": table(lambda x: pearsonr(dt["gs"], x)[0]), "mean similarity": table(lambda x: x.mean())}, axis=1).round(4)

Pearson                  mean similarity                 
                      NLTK   spaCy FreeLing            NLTK   spaCy FreeLing
words               0.4507  0.4606   0.4393          0.5002  0.5053   0.4864
lemmas              0.4494  0.4784   0.4433          0.5149  0.5506   0.5331
Lesk senses         0.4153  0.4406   0.3805          0.4241  0.4324   0.4215
Lesk senses + rest  0.4016  0.4288   0.3787          0.4786  0.5067   0.4868
MFS senses          0.4926  0.4986   0.4562          0.5341  0.5439   0.5288
MFS senses + rest   0.4488  0.4729   0.4357          0.5319  0.5556   0.5402
UKB senses             NaN     NaN   0.4164             NaN     NaN   0.5018
UKB senses + rest      NaN     NaN   0.4145             NaN     NaN   0.5294

## Senses vs. lemmas, pair by pair

As in Lab 3, count the pairs whose similarity increases or decreases with senses compared with lemmas, and look at the pairs with the largest changes. The comparison uses "senses + rest", so every change comes from the disambiguation.

In [9]:
# Pairs whose similarity increases / decreases / stays equal with "senses + rest", compared with lemmas.
rows = []
for pipe, a in senses:
    diff = dt[f"{pipe} {a} senses + rest"] - dt[f"{pipe} lemmas"]
    rows.append({"pipeline": pipe, "algorithm": a, "higher": (diff > 0).sum(), "lower": (diff < 0).sum(), "equal": (diff == 0).sum()})
pd.DataFrame(rows).set_index(["pipeline", "algorithm"])

higher  lower  equal
pipeline algorithm                      
NLTK     Lesk           52    187    220
         MFS           116     34    309
spaCy    Lesk           36    200    223
         MFS            82     59    318
FreeLing Lesk           36    199    224
         MFS            67     30    362
         UKB            88    107    264

### Largest increases and decreases

Lesk and MFS use the spaCy pipeline, and UKB the FreeLing one. Many pairs share the same first sentence (Lab 3), so only the largest change of each first sentence is shown.

In [10]:
def show_pair(i, pipe):
    """Print a sentence pair with its gold score, similarities, lemma set and "senses + rest" sets."""
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    print(f"  S1: {dt[0][i]}")
    print(f"  S2: {dt[1][i]}")
    print("  " + "   ".join(f"{r} {dt[f'{pipe} {r}'][i]:.3f}" for r in representations if f"{pipe} {r}" in dt))
    print(f"  FreeLing lemmas {dt['FreeLing lemmas'][i]:.3f}   UKB senses + rest {dt['FreeLing UKB senses + rest'][i]:.3f}")
    sets = [("Lemmas", {l for _, l, _ in tokens1[pipe][i]}, {l for _, l, _ in tokens2[pipe][i]})]
    sets += [(f"{a} + rest", sense_set(senses[pipe, a][0][i], True), sense_set(senses[pipe, a][1][i], True)) for p, a in senses if p == pipe]
    for name, set1, set2 in sets:
        print(f"  {name}: shared {sorted(set1 & set2)}")
        print(f"  {'':<{len(name)}}  only S1 {sorted(set1 - set2)}")
        print(f"  {'':<{len(name)}}  only S2 {sorted(set2 - set1)}")
    print()

for pipe, a in [("spaCy", "Lesk"), ("spaCy", "MFS"), ("FreeLing", "UKB")]:
    x = dt.assign(diff=dt[f"{pipe} {a} senses + rest"] - dt[f"{pipe} lemmas"])
    increases = x.sort_values("diff", ascending=False, kind="stable").drop_duplicates(subset=[0])
    decreases = x.sort_values("diff", kind="stable").drop_duplicates(subset=[0])
    print(f"=== {a} ({pipe}): largest increases ===\n")
    for i in increases.index[:3]:
        show_pair(i, pipe)
    print(f"=== {a} ({pipe}): largest decreases ===\n")
    for i in decreases.index[:3]:
        show_pair(i, pipe)

=== Lesk (spaCy): largest increases ===

Pair 260  (gold 4.75)
  S1: Van Orden Report (A5-0241/2000)
  S2: Report Van Orden (A5-0241/2000)
  words 1.000   lemmas 0.778   Lesk senses 1.000   Lesk senses + rest 1.000   MFS senses 1.000   MFS senses + rest 1.000
  FreeLing lemmas 0.600   UKB senses + rest 0.600
  Lemmas: shared ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'Van']
          only S1 ['Report']
          only S2 ['report']
  Lesk + rest: shared ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'avant-garde.n.01', 'report.n.01']
               only S1 []
               only S2 []
  MFS + rest: shared ['(', ')', '-', '0241/2000', 'A5', 'Orden', 'avant-garde.n.01', 'report.n.01']
              only S1 []
              only S2 []

Pair 356  (gold 4.0)
  S1: Then perhaps we could have avoided a catastrophe.
  S2: We would maybe then have been able to avoid a disaster.
  words 0.167   lemmas 0.400   Lesk senses 0.800   Lesk senses + rest 0.615   MFS senses 0.500   MFS senses + rest 0.500


## Same lemma, different sense

Lesk and UKB disambiguate each sentence on its own, so a word that appears in both sentences of a pair can get a different sense in each one. MFS cannot do this, because its sense only depends on the lemma and the category.

In [11]:
# Lemmas are lowercased here, as WordNet does when it looks them up.
def shared_lemma_senses(x1, x2):
    """(lemma, category) -> (sense in S1, sense in S2), for the lemmas disambiguated in both sentences."""
    s1 = {(lemma.lower(), ss.pos()): ss for _, lemma, ss in x1 if ss}
    s2 = {(lemma.lower(), ss.pos()): ss for _, lemma, ss in x2 if ss}
    return {k: (s1[k], s2[k]) for k in s1.keys() & s2.keys()}

shared = {key: [shared_lemma_senses(x1, x2) for x1, x2 in zip(*senses[key])] for key in senses}
rows = []
for (pipe, a), pairs in shared.items():
    total = sum(len(s) for s in pairs)
    different = sum(ss1 != ss2 for s in pairs for ss1, ss2 in s.values())
    rows.append({"pipeline": pipe, "algorithm": a, "lemmas in both sentences": total,
                 "different sense": different, "%": round(100 * different / total, 1),
                 "pairs affected": sum(any(ss1 != ss2 for ss1, ss2 in s.values()) for s in pairs)})
pd.DataFrame(rows).set_index(["pipeline", "algorithm"])

lemmas in both sentences  different sense     %  \
pipeline algorithm                                                    
NLTK     Lesk                           1532              324  21.1   
         MFS                            1533                0   0.0   
spaCy    Lesk                           1417              291  20.5   
         MFS                            1418                0   0.0   
FreeLing Lesk                           1582              320  20.2   
         MFS                            1582                0   0.0   
         UKB                            1519               93   6.1   

                    pairs affected  
pipeline algorithm                  
NLTK     Lesk                  206  
         MFS                     0  
spaCy    Lesk                  188  
         MFS                     0  
FreeLing Lesk                  189  
         MFS                     0  
         UKB                    84

In [12]:
# Lemmas that most often get a different Lesk sense in the two sentences of a pair (spaCy pipeline).
mismatches = Counter()
examples = {}
for i, s in enumerate(shared["spaCy", "Lesk"]):
    for lemma, (ss1, ss2) in s.items():
        if ss1 != ss2:
            mismatches[lemma[0]] += 1
            examples.setdefault(lemma[0], (i, ss1, ss2))

for lemma, count in mismatches.most_common(6):
    i, ss1, ss2 = examples[lemma]
    print(f"{lemma:<12} {count:>3} pairs   e.g. pair {i}: {ss1.name()} / {ss2.name()}")

# The senses chosen by Lesk for "report" and "vote" in one of these pairs, with their definitions.
print()
for i, lemma in [(examples["report"][0], "report"), (examples["vote"][0], "vote")]:
    for sentence, x in [(dt[0][i], senses["spaCy", "Lesk"][0][i]), (dt[1][i], senses["spaCy", "Lesk"][1][i])]:
        for word, l, ss in x:
            if ss and l.lower() == lemma:
                print(f"{sentence}\n  {word} -> {ss.name()}: {ss.definition()}")
    print()

# The senses chosen by UKB for "report" in the same pair.
i = examples["report"][0]
for side in (0, 1):
    for word, lemma, ss in senses["FreeLing", "UKB"][side][i]:
        if ss and lemma == "report":
            print(f"UKB, S{side + 1}: {word} -> {ss.name()}: {ss.definition()}")

report        26 pairs   e.g. pair 18: report.n.02 / report.n.01
reading       17 pairs   e.g. pair 17: reading.n.03 / reading.n.01
grass         17 pairs   e.g. pair 28: grass.n.01 / eatage.n.01
act           17 pairs   e.g. pair 65: work.v.03 / act.v.02
union         15 pairs   e.g. pair 79: union.n.01 / coupling.n.03
amendment     14 pairs   e.g. pair 1: amendment.n.01 / amendment.n.02

Mr President, the Cashman report can be summarised in four words: citizens' power over bureaucracy.
  report -> report.n.02: the act of informing by verbal report
Mr President, the Cashman report can be summed up in a few words: the power of citizens on bureaucracy.
  report -> report.n.01: a written document describing the findings of some individual or group

The vote will take place today at 5.30 p.m.
  vote -> vote.n.01: a choice that is made by counting the number of people in favor of each alternative
The vote will place to 17h30.
  vote -> right_to_vote.n.01: a legal right guaranteed by the 15

## Synonyms merged by senses

Different lemmas can end in the same synset. List the lemmas that are merged most often by each algorithm (spaCy for Lesk and MFS, FreeLing for UKB).

In [13]:
for pipe, a in [("spaCy", "Lesk"), ("spaCy", "MFS"), ("FreeLing", "UKB")]:
    merged = Counter()
    for x1, x2 in zip(*senses[pipe, a]):
        by_synset1 = {ss: lemma.lower() for _, lemma, ss in x1 if ss}
        by_synset2 = {ss: lemma.lower() for _, lemma, ss in x2 if ss}
        for ss in by_synset1.keys() & by_synset2.keys():
            if by_synset1[ss] != by_synset2[ss]:
                merged[(by_synset1[ss], by_synset2[ss], ss.name())] += 1
    print(f"{a} ({pipe}): different lemmas mapped to the same synset in {sum(merged.values())} cases")
    for (l1, l2, name), count in merged.most_common(6):
        print(f"  {l1} / {l2} -> {name} ({count} pairs)")
    print()

Lesk (spaCy): different lemmas mapped to the same synset in 69 cases
  catastrophe / disaster -> catastrophe.n.02 (14 pairs)
  hence / therefore -> therefore.r.01 (14 pairs)
  scarcely / hardly -> barely.r.01 (13 pairs)
  summarise / sum -> summarize.v.02 (7 pairs)
  summarise / summarize -> summarize.v.02 (5 pairs)
  mr / mister -> mister.n.01 (4 pairs)

MFS (spaCy): different lemmas mapped to the same synset in 72 cases
  big / large -> large.a.01 (14 pairs)
  hence / therefore -> therefore.r.01 (14 pairs)
  scarcely / hardly -> barely.r.01 (13 pairs)
  summarise / sum -> summarize.v.02 (7 pairs)
  fervent / ardent -> ardent.s.01 (5 pairs)
  mr / mister -> mister.n.01 (4 pairs)

UKB (FreeLing): different lemmas mapped to the same synset in 91 cases
  catastrophe / disaster -> calamity.n.01 (14 pairs)
  big / large -> large.a.01 (14 pairs)
  hence / therefore -> therefore.r.01 (14 pairs)
  scarcely / hardly -> hardly.r.02 (13 pairs)
  quite / rather -> quite.r.01 (9 pairs)
  change / 

## Questions

### Results

| Representation | NLTK | spaCy | FreeLing |
|---|---|---|---|
| Words (Lab 2) | 0.4507 | 0.4606 | 0.4393 |
| Lemmas (Lab 3) | 0.4494 | 0.4784 | 0.4433 |
| Lesk senses | 0.4153 | 0.4406 | 0.3805 |
| Lesk senses + rest | 0.4016 | 0.4288 | 0.3787 |
| **MFS senses** | **0.4926** | **0.4986** | **0.4562** |
| MFS senses + rest | 0.4488 | 0.4729 | 0.4357 |
| UKB senses | | | 0.4164 |
| UKB senses + rest | | | 0.4145 |

(Pearson correlation with the gold standard.)

- **The conclusions do not depend on the library.** In the three pipelines MFS senses are the best representation, and the two Lesk rows are the worst, below words and lemmas.
- **UKB is between Lesk and MFS.** On the same FreeLing tokens, UKB senses (0.4164) are above Lesk senses (0.3805), but below MFS senses (0.4562) and below lemmas (0.4433).
- **spaCy is better than NLTK in every row, and FreeLing is the lowest in every row.** The POS tags of spaCy are more accurate (in pair 12 NLTK tags *adopts* as a plural noun, so it is not reduced to *adopt* and is looked up as a noun), and its lemmas are lowercased (the NLTK lemma of *The* is *The*). With NLTK, lemmas are not even better than words (0.4494 vs 0.4507). So the submitted comparison of UKB (FreeLing) with Lesk (spaCy) is not fair to UKB: part of the difference comes from the preprocessing.
- **Adding the words without a sense lowers the correlation** in all cases (for example 0.4986 → 0.4729 for spaCy MFS). They are mostly function words and punctuation, which are shared by most pairs and do not carry the meaning, as in Lab 2.

### Effect of the disambiguation, pair by pair

"Senses + rest" only differs from lemmas in the words that get a sense, so comparing them shows the effect of the disambiguation alone:

| | higher | lower | equal |
|---|---|---|---|
| NLTK Lesk | 52 | 187 | 220 |
| NLTK MFS | 116 | 34 | 309 |
| spaCy Lesk | 36 | 200 | 223 |
| spaCy MFS | 82 | 59 | 318 |
| FreeLing Lesk | 36 | 199 | 224 |
| FreeLing MFS | 67 | 30 | 362 |
| FreeLing UKB | 88 | 107 | 264 |

**Lesk lowers the similarity of many more pairs than it raises, MFS the opposite, and UKB is balanced.** The examples show the causes:

- **Increase: different lemmas get the same synset.** Pair 260: *Report* / *report* are different lemmas (spaCy keeps the capital letter), but WordNet ignores the case and both get `report.n.01` (0.778 → 1.000). Pair 7: *fishermen* / *fishers* both get `fisherman.n.01` (0.800 → 1.000). Pair 356: Lesk gives *catastrophe* / *disaster* the same synset `catastrophe.n.02`, and UKB `calamity.n.01`, but MFS does not (`calamity.n.01` / `catastrophe.n.02`). UKB also merges *change* / *modification* in pair 42 (0.769 → 0.917) and *whole* / *unit* in pair 219 (0.364 → 0.500).
- **Lesk decrease: the same lemma gets a different sense in each sentence.** Pair 12: the lemmas are identical (1.000), but Lesk gives *adopt* `espouse.v.03` / `adopt.v.01` and *resolution* `resolving_power.n.01` / `resolution.n.01` (→ 0.556), while MFS and UKB keep 1.000. Pair 209: *whole* gets `whole.n.02` / `whole.n.01` (0.556 → 0.273). Pair 187: *aid*, *market*, *grass* and *essential* get a different sense in each sentence (0.412 → 0.143), while MFS and UKB stay at 0.412.
- **UKB decrease: the same cause, less often.** In pair 111 *place* gets `topographic_point.n.01` / `position.n.06` (0.889 → 0.700), and in pair 37 *qualify* gets `qualify.v.04` / `modify.v.02` (0.500 → 0.364). In pair 37 UKB also gives *House* (the Parliament) the sense `theater.n.01`.
- **MFS decrease: the same word gets a different POS tag.** MFS always gives the same sense to the same lemma and category, so it can only lose a match when the tagger changes the category. Pair 197: *inactive* is an adjective in S1 (`inactive.s.01`), but in S2 the word order changes, it is not tagged as an adjective and gets no sense (1.000 → 0.800). Pair 273: *separate* is a verb in S1 (`separate.v.01`) and an adjective in S2 (`separate.a.01`).

### Why Lesk does not help

**About 20% of the lemmas that appear in both sentences of a pair get a different Lesk sense** (324 of 1532 with NLTK, 291 of 1417 with spaCy, 320 of 1582 with FreeLing), which affects 206, 188 and 189 of the 459 pairs. With MFS this never happens. Lesk disambiguates each sentence separately, using only its own words as context, so a small change in the wording can change the sense. The lemma that gets different senses most often is *report* (26 pairs): in pair 18 it gets `report.n.02` ("the act of informing by verbal report") in one sentence and `report.n.01` ("a written document...") in the other.

The senses are also often wrong, because the sentences are short and the definitions share few words with them: in *"The vote will place to 17h30."* *vote* gets `right_to_vote.n.01`, and in the first sentence of the corpus *chance* gets `probability.n.01` and *seize* gets `assume.v.06`.

### Why UKB is better than Lesk, but not than MFS

UKB uses the whole WordNet graph and is **much more consistent than Lesk**: only 6.1% of the shared lemmas get a different sense (93 of 1519), which affects 84 pairs. It also merges more synonyms than the other two algorithms (91 cases, against 69 for Lesk and 72 for MFS). But it is still less consistent than MFS, and consistent is not correct: in pair 18 *report* gets `report.n.03` in both sentences, which is "a short account of the news".

### Why MFS helps

MFS keeps the advantage of lemmas (the same lemma always gives the same item) and adds a new one: **synonyms are mapped to the same synset**, such as *big* / *large* → `large.a.01`, *hence* / *therefore* → `therefore.r.01` (14 pairs each) and *scarcely* / *hardly* → `barely.r.01` (13 pairs). Lesk merges some synonyms too (*catastrophe* / *disaster*, 14 pairs), but loses more matches than it gains.

### Conclusion

Senses can capture synonyms that words and lemmas miss, but only if the disambiguation is consistent between the two sentences. On the same FreeLing tokens, the three algorithms have the same order in correlation and in consistency: MFS (0.4562, 0% different senses), UKB (0.4164, 6.1%) and Lesk (0.3805, 20.2%). An error of the disambiguation does not only choose a wrong sense, it also breaks a match between two words that were equal. So the simple MFS baseline is the best representation with the three libraries. Disambiguating both sentences with a common context could reduce this problem.

## Notes

Design decisions and findings from the discussion of this lab, to remember why the notebook is written this way.

1. **Pipeline.** Tokens → POS tags → lemmas → WSD. The lemmatizer needs the POS tag, and WSD needs the POS tag (WordNet category), the lemma (the word looked up) and the sentence (Lesk context). Steps 1-3 can be done with NLTK, spaCy or FreeLing. Lesk and MFS are always NLTK + WordNet, because spaCy has no WordNet, and UKB is always TextServer.
2. **One library per column.** Comparing two representations should change only one thing. The first version mixed NLTK words (Lab 2) with spaCy lemmas and senses, so part of the difference came from the tokenizer. Here each column uses one library from start to end.
3. **Submitted version.** The submitted notebook (team repository `team/`, tag `lab06-submitted`) uses spaCy for words, lemmas and Lesk, TextServer for UKB and its FreeLing lemmas, and no MFS, because the statement only presents Lesk and UKB. This version is only for study.
4. **Case.** spaCy lemmas are lowercase except proper nouns and *I* (863 tokens); NLTK lemmas keep the original case (1627 tokens, mostly sentence-initial words such as *The*); FreeLing lemmas are lowercase except document codes (64 tokens). No extra lowercasing is applied, because names should keep their capital letter. WordNet lowercases the word before looking it up, so the senses do not depend on the case. The analysis cells lowercase the lemmas only to decide whether two words are "the same lemma", as WordNet does.
5. **POS tagger vs. lemmatizer.** `wn.synsets` reduces inflected forms by itself (morphy), so for WSD the lemmatizer matters less than the POS tag, which decides the category. The MFS decreases (pairs 197 and 273) come from POS tags.
6. **Lesk and satellite adjectives.** `lesk(context, word, pos)` filters the synsets by `ss.pos() == pos`, so adjectives whose synsets are all satellites (`s`) get no sense: 146 words in the spaCy pipeline (*transferable*, *fervent*, *selective*...), while MFS finds a sense for them. Passing `synsets=wn.synsets(word, pos)` fixes it. The submitted version does not have this fix (its spaCy Lesk senses give 0.4467 instead of 0.4406).
7. **NLTK Lesk is very simple.** It splits the definition on spaces, does not lowercase or lemmatize it, does not remove stopwords, ignores the examples, and returns the first synset on a tie (so with no overlap it returns the first sense, like MFS).
8. **Senses vs. senses + rest.** Senses changes two things at once compared with lemmas: it replaces the lemmas by synsets, and it leaves out the words without a sense. "Senses + rest" only changes the first, so it is used for the pair-by-pair analysis; "senses" vs. "senses + rest" in the table gives the effect of the second.
9. **Repeated pairs.** Many pairs share the same first sentence (Lab 3), so the examples keep only the largest change of each first sentence, with a stable sort so that ties keep the lowest pair number.
10. **UKB and FreeLing.** TextServer gives the tokens, lemmas and senses together, so UKB cannot be run on the spaCy tokens. Instead, Lesk and MFS are also run on the FreeLing tokens, to compare the three algorithms on the same tokens.
11. **Empty sets.** With FreeLing, pair 260 is a single multiword noun without a sense in each sentence (`van_orden_report` / `report_van_orden`), so both sets of senses are empty. `jaccard_distance` fails with two empty sets, so `jaccard` returns 0 in this case.
